# v9_b Step 2: Candidate–parent and authentic-standard MS2 annotation

This workflow resolves input and output paths from the v9_b working directory and performs three linked analyses:

1. candidate–parent PRM MS2 modified-cosine comparison;
2. authentic-standard MS1/MS2 annotation and RT-isomer disambiguation;
3. integration of both evidence types into the complete v9 table-04 candidate set.

Candidate spectra are extracted once from the 83 PRM mzML files and reused by both MS2 analyses. Pellets and supernatant remain separate because `matrix` is part of the candidate key.

## 1. Imports, paths and analysis parameters


In [ ]:
from pathlib import Path
from collections import defaultdict
import math
import os
import re

os.environ.setdefault("MPLCONFIGDIR", "/tmp/candidate_filter_annotation_mpl")
os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/candidate_filter_annotation_numba")

import numpy as np
import pandas as pd
from pyteomics import mzml
from matchms import Spectrum
from matchms.filtering import normalize_intensities
from matchms.similarity import ModifiedCosineHungarian


PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "results" / "step1_filtered_results").is_dir():
    raise FileNotFoundError("Run this notebook from v9_b after Step 1")
INPUT_DIR = PROJECT_DIR / "input" / "step2_candidate_filter_annotation"
CANDIDATE_DIR = PROJECT_DIR / "results" / "step1_filtered_results"
PRM_LIST_ROOT = INPUT_DIR / "prm_target_lists"
MZML_ROOT = INPUT_DIR / "mzML"
STANDARD_TABLE = INPUT_DIR / "authentic_standards" / "Authentic_standard_list.csv"
RESULT_DIR = PROJECT_DIR / "results" / "step2_candidate_filter_annotation"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

GROUPS = ["DnCl_pellets", "DnCl_supernatant", "DnHz_pellets", "DnHz_supernatant"]
STEROIDS = ["Estradiol", "Estrone", "DHEA", "Pregnenolone", "Progesterone"]
NCE_VALUES = (25, 35)

PRECURSOR_TOLERANCE_DA = 0.01
SPECTRUM_RT_WINDOW_S = 3.0
MIN_RAW_PEAKS = 4
FRAGMENT_TOLERANCE_DA = 0.02
MIN_RELATIVE_INTENSITY = 0.01
PRECURSOR_EXCLUSION_DA = 1.5
MZ_POWER = 0.0
INTENSITY_POWER = 0.5

PARENT_MIN_MODIFIED_COSINE = 0.4
PARENT_MIN_MATCHED_PEAKS = 4

AUTHENTIC_MS1_PPM_TOLERANCE = 10.0
AUTHENTIC_MS1_RT_TOLERANCE_S = 4.0
AUTHENTIC_MIN_MODIFIED_COSINE = 0.4
AUTHENTIC_MIN_MATCHED_PEAKS = 4
AUTHENTIC_UNRESOLVED_RT_S = 3.0

CANDIDATE_KEY = ["steroid", "derivatization", "matrix", "enzyme", "unlabeled_feature"]

print(f"Project directory: {PROJECT_DIR}")
print(f"Input directory: {INPUT_DIR}")
print(f"Result directory: {RESULT_DIR}")

### Parameters

- PRM precursor routing tolerance: 0.01 Da
- Preferred candidate/parent scan window: target RT ±3 s; otherwise the best scan within the scheduled interval
- Minimum raw/filtered peaks: 4
- Relative-intensity filter: 1%
- Precursor exclusion: ±1.5 Da
- Fragment tolerance: 0.02 Da
- Modified cosine: Hungarian assignment, `mz_power=0`, `intensity_power=0.5`
- Parent support: modified cosine ≥0.4 and matched peaks ≥4 at either 25 or 35 NCE
- Authentic-standard MS1: derivatization matched, ≤10 ppm and |ΔRT|≤4 s
- Authentic-standard MS2: modified cosine >0.4 and matched peaks ≥4
- Authentic-standard RT disambiguation: all standards within 3 s are retained; otherwise retain the smallest |ΔRT|

## 2. Load the ten v9 table-04 candidate files and authentic-standard metadata

In [ ]:
def infer_candidate_metadata(path):
    lower_name = path.name.lower()
    steroid = next(name for name in STEROIDS if f"_{name.lower()}_" in f"_{lower_name}_")
    derivatization = "DnCl" if "dncl" in lower_name else "DnHz"
    matrix = "supernatant" if "supernatant" in lower_name else "pellets"
    return steroid, derivatization, matrix


def format_enzyme_label(value):
    '''Use standard CYP nomenclature while preserving WT when present.'''
    label = str(value)
    if label == "WT" or label.startswith("CYP"):
        return label
    return f"CYP{label}"


candidate_paths = sorted(CANDIDATE_DIR.glob("04_*unique_enzyme_specific_unlabeled_features.tsv"))
assert len(candidate_paths) == 10, f"Expected 10 v9 table-04 inputs, found {len(candidate_paths)}"

candidate_frames = []
candidate_manifest_rows = []
required_candidate_columns = [
    "unlabeled_feature",
    "unlabeled_feature_mz",
    "unlabeled_feature_rtime",
    "enzyme",
]
for path in candidate_paths:
    steroid, derivatization, matrix = infer_candidate_metadata(path)
    frame = pd.read_csv(path, sep="\t", usecols=required_candidate_columns)
    if list(frame.columns) != required_candidate_columns:
        raise ValueError(f"Unexpected columns in {path.name}: {list(frame.columns)}")
    frame["enzyme"] = frame["enzyme"].map(format_enzyme_label)
    frame.insert(0, "matrix", matrix)
    frame.insert(0, "derivatization", derivatization)
    frame.insert(0, "steroid", steroid)
    candidate_frames.append(frame)
    candidate_manifest_rows.append({
        "steroid": steroid,
        "derivatization": derivatization,
        "matrix": matrix,
        "candidate_rows": len(frame),
        "source_file": path.name,
    })

base_candidates = pd.concat(candidate_frames, ignore_index=True)
assert len(base_candidates) == 1591, f"Expected 1,591 candidates, found {len(base_candidates)}"
assert not base_candidates.duplicated(CANDIDATE_KEY).any(), "Duplicate candidate keys in table-04 inputs"

standards = pd.read_csv(STANDARD_TABLE, encoding="utf-8-sig")
standards = standards.loc[:, ~standards.columns.astype(str).str.startswith("Unnamed")].copy()
standards = standards.rename(columns={
    "No": "authentic_standard_id",
    "Name": "authentic_standard_name",
    "Derivatization": "derivatization",
    "Exact_mass_[M+H]": "authentic_standard_mz",
    "Exact_mass_[M+H]+_after_derivatization": "authentic_standard_mz",
    "RT_min": "RT",
})
required_standard_columns = {
    "authentic_standard_id", "derivatization", "authentic_standard_name",
    "KEGG_number", "authentic_standard_mz", "RT",
}
missing_standard_columns = required_standard_columns.difference(standards.columns)
if missing_standard_columns:
    raise KeyError(f"Missing authentic-standard columns: {sorted(missing_standard_columns)}")
assert not standards.empty, "Authentic-standard table is empty"
assert standards["authentic_standard_id"].is_unique, "Authentic-standard IDs must be unique"

standard_rt_rows = []
for row in standards.itertuples(index=False):
    for isomer_index, rt_text in enumerate(str(row.RT).split(";"), start=1):
        rt_min = float(rt_text.strip())
        standard_rt_rows.append({
            "authentic_standard_id": row.authentic_standard_id,
            "derivatization": row.derivatization,
            "authentic_standard_name": row.authentic_standard_name,
            "KEGG_number": row.KEGG_number,
            "authentic_standard_mz": float(row.authentic_standard_mz),
            "standard_rt_isomer": isomer_index,
            "standard_rt_min": rt_min,
            "standard_rt_s": rt_min * 60.0,
        })
standard_rts = pd.DataFrame(standard_rt_rows)

display(pd.DataFrame(candidate_manifest_rows).sort_values(["derivatization", "matrix", "steroid"]))
print(f"Combined v9 candidates: {len(base_candidates):,}")
print(f"Authentic standards: {len(standards)}; listed derivative RT isomers: {len(standard_rts)}")

## 3. Discover and validate the 83 PRM target-list/mzML pairs

In [ ]:
def parse_group(group):
    return group.split("_", 1)


def parse_target_list_metadata(csv_path, group):
    derivatization, matrix = parse_group(group)
    stem = csv_path.name.removesuffix("_PRM_MS2.csv")
    enzyme = format_enzyme_label(stem.split("_", 1)[0])
    steroid = next(name for name in STEROIDS if f"_{name}_" in f"_{stem}_")
    part_match = re.search(r"_part(?P<part>\d+)_of_(?P<total>\d+)_", stem)
    return {
        "group": group,
        "derivatization": derivatization,
        "matrix": matrix,
        "enzyme": enzyme,
        "steroid": steroid,
        "part": int(part_match.group("part")) if part_match else 1,
        "total_parts": int(part_match.group("total")) if part_match else 1,
        "target_list_id": stem,
    }


def discover_run_pairs():
    rows = []
    for group in GROUPS:
        input_dir = PRM_LIST_ROOT / f"{group}_all_input_lists"
        mzml_dir = MZML_ROOT / f"mzML_{group}_MS2"
        for csv_path in sorted(input_dir.glob("*.csv")):
            stem = csv_path.name.removesuffix("_PRM_MS2.csv")
            mzml_path = mzml_dir / f"{stem}_PRM_MS2.mzML"
            if not mzml_path.exists():
                raise FileNotFoundError(f"Missing paired mzML for {csv_path.name}")
            rows.append({
                **parse_target_list_metadata(csv_path, group),
                "csv_path": csv_path,
                "mzml_path": mzml_path,
            })
    return pd.DataFrame(rows)


run_manifest = discover_run_pairs()
assert len(run_manifest) == 83, f"Expected 83 run pairs, found {len(run_manifest)}"
display(run_manifest.groupby(["derivatization", "matrix"]).size().rename("run_count").reset_index())

## 4. Shared MS2 extraction and preprocessing functions

In [ ]:
def read_targets(csv_path):
    targets = pd.read_csv(csv_path)
    required = {"Mass [m/z]", "Start [min]", "End [min]", "(N)CE", "Comment"}
    missing = required - set(targets.columns)
    if missing:
        raise ValueError(f"Missing columns in {csv_path.name}: {sorted(missing)}")
    targets = targets.reset_index(drop=True).rename_axis("target_index").reset_index()
    targets["target_mz"] = targets["Mass [m/z]"].astype(float)
    targets["start_min"] = targets["Start [min]"].astype(float)
    targets["end_min"] = targets["End [min]"].astype(float)
    targets["target_rt_min"] = (targets["start_min"] + targets["end_min"]) / 2.0
    targets["nce"] = targets["(N)CE"].astype(float).round().astype(int)
    targets["feature_id"] = targets["Comment"].astype(str)
    targets["target_type"] = np.where(
        targets["feature_id"].str.contains("parent", case=False),
        "parent",
        "candidate",
    )
    return targets


def spectrum_metadata(spec):
    try:
        precursor = spec["precursorList"]["precursor"][0]
        precursor_mz = float(precursor["selectedIonList"]["selectedIon"][0]["selected ion m/z"])
        nce = int(round(float(precursor.get("activation", {}).get("collision energy"))))
        rt_min = float(spec["scanList"]["scan"][0]["scan start time"])
    except (KeyError, IndexError, TypeError, ValueError):
        return None
    return precursor_mz, nce, rt_min


def extract_best_scans(mzml_path, targets, require_scheduled_interval=True):
    target_bins = defaultdict(list)
    for target in targets.to_dict("records"):
        target_bins[(target["nce"], math.floor(target["target_mz"] * 100))].append(target)

    best = {}
    with mzml.MzML(str(mzml_path)) as reader:
        for scan_index, spec in enumerate(reader):
            if spec.get("ms level") != 2:
                continue
            metadata = spectrum_metadata(spec)
            if metadata is None:
                continue
            precursor_mz, nce, rt_min = metadata
            mz_array = np.asarray(spec.get("m/z array", []), dtype=float)
            intensity_array = np.asarray(spec.get("intensity array", []), dtype=float)
            if len(mz_array) < MIN_RAW_PEAKS:
                continue
            tic = float(intensity_array.sum())
            mz_bin = math.floor(precursor_mz * 100)

            for nearby_bin in (mz_bin - 1, mz_bin, mz_bin + 1):
                for target in target_bins.get((nce, nearby_bin), []):
                    if abs(precursor_mz - target["target_mz"]) > PRECURSOR_TOLERANCE_DA:
                        continue
                    if require_scheduled_interval and not (
                        target["start_min"] <= rt_min <= target["end_min"]
                    ):
                        continue
                    rt_error_s = (rt_min - target["target_rt_min"]) * 60.0
                    if not require_scheduled_interval and abs(rt_error_s) > SPECTRUM_RT_WINDOW_S:
                        continue
                    within_preferred_rt = abs(rt_error_s) <= SPECTRUM_RT_WINDOW_S
                    priority = (int(within_preferred_rt), tic, -abs(rt_error_s))
                    previous = best.get(target["target_index"])
                    if previous is None or priority > previous["priority"]:
                        best[target["target_index"]] = {
                            "priority": priority,
                            "scan_index": scan_index,
                            "precursor_mz": precursor_mz,
                            "rt_min": rt_min,
                            "rt_error_s": rt_error_s,
                            "within_preferred_rt_window": within_preferred_rt,
                            "tic": tic,
                            "raw_peak_count": len(mz_array),
                            "mz": mz_array.copy(),
                            "intensity": intensity_array.copy(),
                        }
    return best


def build_filtered_spectrum(scan):
    if scan is None:
        return None, 0
    mz_values = scan["mz"]
    intensities = scan["intensity"]
    if len(mz_values) == 0 or float(intensities.max(initial=0.0)) <= 0:
        return None, 0
    keep = intensities >= float(intensities.max()) * MIN_RELATIVE_INTENSITY
    keep &= np.abs(mz_values - scan["precursor_mz"]) > PRECURSOR_EXCLUSION_DA
    keep &= np.isfinite(mz_values) & np.isfinite(intensities) & (intensities > 0)
    mz_values = mz_values[keep]
    intensities = intensities[keep]
    if len(mz_values) < MIN_RAW_PEAKS:
        return None, len(mz_values)
    order = np.argsort(mz_values)
    spectrum = Spectrum(
        mz=mz_values[order],
        intensities=intensities[order],
        metadata={"precursor_mz": float(scan["precursor_mz"])},
    )
    return normalize_intensities(spectrum), len(mz_values)


modified_cosine = ModifiedCosineHungarian(
    tolerance=FRAGMENT_TOLERANCE_DA,
    mz_power=MZ_POWER,
    intensity_power=INTENSITY_POWER,
)

## 5. Extract candidate and parent spectra once, then calculate candidate–parent similarity

The `candidate_spectra` cache is retained for the authentic-standard comparison below. This avoids a second pass through the 83 candidate PRM mzML files.

In [ ]:
candidate_spectra = {}
candidate_target_keys = set()
parent_score_rows = []

for run_number, run in enumerate(run_manifest.itertuples(index=False), start=1):
    targets = read_targets(run.csv_path)
    best_scans = extract_best_scans(run.mzml_path, targets, require_scheduled_interval=True)

    parent_rows = targets.loc[targets["target_type"].eq("parent")]
    if set(parent_rows["nce"]) != set(NCE_VALUES):
        raise ValueError(f"Expected parent NCE 25 and 35 in {run.csv_path.name}")

    parent_spectra = {}
    for parent in parent_rows.itertuples(index=False):
        parent_scan = best_scans.get(parent.target_index)
        parent_spectrum, parent_filtered_count = build_filtered_spectrum(parent_scan)
        parent_spectra[parent.nce] = (parent_spectrum, parent_filtered_count, parent_scan)

    for candidate in targets.loc[targets["target_type"].eq("candidate")].itertuples(index=False):
        base_key = (
            run.steroid,
            run.derivatization,
            run.matrix,
            run.enzyme,
            candidate.feature_id,
        )
        spectrum_key = (*base_key, candidate.nce)
        if spectrum_key in candidate_spectra:
            raise AssertionError(f"Duplicate candidate/NCE target across PRM lists: {spectrum_key}")
        candidate_target_keys.add(base_key)

        candidate_scan = best_scans.get(candidate.target_index)
        candidate_spectrum, candidate_filtered_count = build_filtered_spectrum(candidate_scan)
        candidate_spectra[spectrum_key] = candidate_spectrum

        parent_spectrum, parent_filtered_count, parent_scan = parent_spectra.get(
            candidate.nce,
            (None, 0, None),
        )
        row = {
            "steroid": run.steroid,
            "derivatization": run.derivatization,
            "matrix": run.matrix,
            "enzyme": run.enzyme,
            "candidate_feature": candidate.feature_id,
            "nce": candidate.nce,
        }

        if candidate_scan is None:
            parent_score_rows.append({
                **row,
                "status": "candidate_ms2_missing",
                "ms2_supported": False,
            })
            continue
        if parent_scan is None:
            parent_score_rows.append({
                **row,
                "status": "parent_ms2_missing",
                "ms2_supported": False,
            })
            continue

        qc = {
            "candidate_scan_index": candidate_scan["scan_index"],
            "parent_scan_index": parent_scan["scan_index"],
            "candidate_rt_error_s": candidate_scan["rt_error_s"],
            "parent_rt_error_s": parent_scan["rt_error_s"],
            "candidate_within_best_rt_window": candidate_scan["within_preferred_rt_window"],
            "parent_within_best_rt_window": parent_scan["within_preferred_rt_window"],
            "candidate_raw_peak_count": candidate_scan["raw_peak_count"],
            "parent_raw_peak_count": parent_scan["raw_peak_count"],
            "candidate_filtered_peak_count": candidate_filtered_count,
            "parent_filtered_peak_count": parent_filtered_count,
        }
        if candidate_spectrum is None:
            parent_score_rows.append({
                **row,
                **qc,
                "status": "insufficient_candidate_peaks_after_filter",
                "ms2_supported": False,
            })
            continue
        if parent_spectrum is None:
            parent_score_rows.append({
                **row,
                **qc,
                "status": "insufficient_parent_peaks_after_filter",
                "ms2_supported": False,
            })
            continue

        score = modified_cosine.pair(parent_spectrum, candidate_spectrum)
        cosine_value = float(score["score"])
        matched_peaks = int(score["matches"])
        supported = (
            cosine_value >= PARENT_MIN_MODIFIED_COSINE
            and matched_peaks >= PARENT_MIN_MATCHED_PEAKS
        )
        parent_score_rows.append({
            **row,
            **qc,
            "status": "ok",
            "modified_cosine": cosine_value,
            "matched_peaks": matched_peaks,
            "ms2_supported": supported,
        })

    if run_number % 10 == 0 or run_number == len(run_manifest):
        print(f"Processed {run_number}/{len(run_manifest)} candidate/parent mzML files")

parent_scores_long = pd.DataFrame(parent_score_rows)
expected_candidate_keys = set(base_candidates[CANDIDATE_KEY].itertuples(index=False, name=None))
assert candidate_target_keys == expected_candidate_keys, (
    f"PRM target-list candidate keys differ from table-04 keys: "
    f"missing={len(expected_candidate_keys - candidate_target_keys)}, "
    f"extra={len(candidate_target_keys - expected_candidate_keys)}"
)

print(f"NCE-level candidate comparisons: {len(parent_scores_long):,}")
print(f"Cached candidate NCE spectra: {sum(value is not None for value in candidate_spectra.values()):,}/{len(candidate_spectra):,}")
display(parent_scores_long["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="rows"))

## 6. Build and write the candidate–parent comparison table

In [ ]:
parent_wide = base_candidates.copy()
parent_merge_columns = [
    "steroid", "derivatization", "matrix", "enzyme", "candidate_feature",
    "status", "modified_cosine", "matched_peaks", "ms2_supported",
    "candidate_scan_index", "parent_scan_index", "candidate_rt_error_s", "parent_rt_error_s",
    "candidate_within_best_rt_window", "parent_within_best_rt_window",
    "candidate_raw_peak_count", "parent_raw_peak_count",
    "candidate_filtered_peak_count", "parent_filtered_peak_count",
]

for nce in NCE_VALUES:
    subset = parent_scores_long.loc[
        parent_scores_long["nce"].eq(nce),
        parent_merge_columns,
    ].copy()
    subset = subset.rename(columns={
        "candidate_feature": "unlabeled_feature",
        **{column: f"{column}_{nce}" for column in parent_merge_columns[5:]},
    })
    parent_wide = parent_wide.merge(
        subset,
        on=CANDIDATE_KEY,
        how="left",
        validate="one_to_one",
    )

for nce in NCE_VALUES:
    parent_wide[f"ms2_supported_{nce}"] = parent_wide[f"ms2_supported_{nce}"].eq(True)

score_25 = parent_wide["modified_cosine_25"].to_numpy(dtype=float)
score_35 = parent_wide["modified_cosine_35"].to_numpy(dtype=float)
choose_35 = np.nan_to_num(score_35, nan=-np.inf) > np.nan_to_num(score_25, nan=-np.inf)
both_missing = np.isnan(score_25) & np.isnan(score_35)

parent_wide["best_modified_cosine"] = np.where(choose_35, score_35, score_25)
parent_wide.loc[both_missing, "best_modified_cosine"] = np.nan
parent_wide["best_nce"] = np.where(choose_35, 35, 25).astype(float)
parent_wide.loc[both_missing, "best_nce"] = np.nan
parent_wide["best_matched_peaks"] = np.where(
    choose_35,
    parent_wide["matched_peaks_35"],
    parent_wide["matched_peaks_25"],
)
parent_wide.loc[both_missing, "best_matched_peaks"] = np.nan
parent_wide["supporting_nce_count"] = parent_wide[
    ["ms2_supported_25", "ms2_supported_35"]
].sum(axis=1).astype(int)
parent_wide["ms2_supported_any_nce"] = parent_wide["supporting_nce_count"].ge(1)
parent_wide["ms2_evaluation_status"] = np.select(
    [
        parent_wide["modified_cosine_25"].notna() & parent_wide["modified_cosine_35"].notna(),
        parent_wide["modified_cosine_25"].notna(),
        parent_wide["modified_cosine_35"].notna(),
    ],
    ["both_nce_evaluable", "25_only_evaluable", "35_only_evaluable"],
    default="not_evaluable",
)

parent_columns = [
    "steroid", "derivatization", "matrix", "enzyme",
    "unlabeled_feature", "unlabeled_feature_mz", "unlabeled_feature_rtime",
    "modified_cosine_25", "matched_peaks_25",
    "modified_cosine_35", "matched_peaks_35",
    "best_modified_cosine", "best_nce", "best_matched_peaks",
    "supporting_nce_count", "ms2_supported_any_nce", "ms2_evaluation_status",
]
parent_output = (
    parent_wide[parent_columns]
    .sort_values(CANDIDATE_KEY)
    .reset_index(drop=True)
)
parent_path = RESULT_DIR / "01_parent_candidate_comparison.tsv"
parent_output.to_csv(parent_path, sep="\t", index=False)

print(f"Wrote: {parent_path.relative_to(PROJECT_DIR)}")
print(f"Parent-supported candidate keys: {int(parent_output['ms2_supported_any_nce'].sum()):,}/{len(parent_output):,}")
display(parent_output.head(10))

## 7. Authentic-standard MS1 m/z and RT matching

In [ ]:
ms1_pair_rows = []
for derivatization, candidate_group in parent_output.groupby("derivatization", sort=False):
    standard_group = standard_rts.loc[standard_rts["derivatization"].eq(derivatization)]
    for candidate in candidate_group.itertuples(index=False):
        mz_error_ppm = (
            (candidate.unlabeled_feature_mz - standard_group["authentic_standard_mz"])
            / standard_group["authentic_standard_mz"]
            * 1e6
        )
        rt_error_s = candidate.unlabeled_feature_rtime - standard_group["standard_rt_s"]
        keep = (
            mz_error_ppm.abs().le(AUTHENTIC_MS1_PPM_TOLERANCE)
            & rt_error_s.abs().le(AUTHENTIC_MS1_RT_TOLERANCE_S)
        )
        for standard_index in standard_group.index[keep]:
            standard = standard_group.loc[standard_index]
            ms1_pair_rows.append({
                **{column: getattr(candidate, column) for column in CANDIDATE_KEY},
                "authentic_standard_id": standard.authentic_standard_id,
                "authentic_standard_name": standard.authentic_standard_name,
                "KEGG_number": standard.KEGG_number,
                "authentic_standard_mz": standard.authentic_standard_mz,
                "standard_rt_isomer": int(standard.standard_rt_isomer),
                "matched_standard_rt_min": standard.standard_rt_min,
                "mz_error_ppm": float(mz_error_ppm.loc[standard_index]),
                "rt_error_s": float(rt_error_s.loc[standard_index]),
            })

ms1_pairs = pd.DataFrame(ms1_pair_rows)
if ms1_pairs.empty:
    raise AssertionError("No candidate–standard pairs passed the MS1 criteria")

matched_standard_ids = set(ms1_pairs["authentic_standard_id"])
print(f"MS1 candidate–standard-isomer pairs: {len(ms1_pairs):,}")
print(f"Unique candidate keys passing MS1: {len(ms1_pairs[CANDIDATE_KEY].drop_duplicates()):,}")
print(f"Authentic standards represented after MS1: {len(matched_standard_ids)}/{len(standards)}")

## 8. Extract authentic-standard spectra and calculate candidate–standard similarity

In [ ]:
def authentic_standard_mzml_path(standard_id, derivatization):
    if derivatization == "DnCl":
        path = MZML_ROOT / "DnCl_AS_MS2_mzML" / f"{standard_id.replace('_', '')}_DnCl_MS2.mzML"
    elif derivatization == "DnHz":
        matches = sorted((MZML_ROOT / "DnHz_AS_MS2_mzML").glob(f"{standard_id}_*_MS2.mzML"))
        if len(matches) != 1:
            raise FileNotFoundError(
                f"Expected one authentic-standard mzML for {standard_id}, found {len(matches)}"
            )
        path = matches[0]
    else:
        raise ValueError(f"Unsupported derivatization: {derivatization}")
    if not path.exists():
        raise FileNotFoundError(path)
    return path


standard_spectra = {}
for standard_id, standard_group in standard_rts.groupby("authentic_standard_id", sort=False):
    derivatization = standard_group["derivatization"].iloc[0]
    target_rows = []
    target_index = 0
    for standard in standard_group.itertuples(index=False):
        for nce in NCE_VALUES:
            target_rows.append({
                "target_index": target_index,
                "target_mz": standard.authentic_standard_mz,
                "target_rt_min": standard.standard_rt_min,
                "start_min": standard.standard_rt_min - SPECTRUM_RT_WINDOW_S / 60.0,
                "end_min": standard.standard_rt_min + SPECTRUM_RT_WINDOW_S / 60.0,
                "nce": nce,
                "standard_rt_isomer": standard.standard_rt_isomer,
            })
            target_index += 1
    targets = pd.DataFrame(target_rows)
    mzml_path = authentic_standard_mzml_path(standard_id, derivatization)
    best_scans = extract_best_scans(mzml_path, targets, require_scheduled_interval=False)
    for target in targets.itertuples(index=False):
        spectrum, _ = build_filtered_spectrum(best_scans.get(target.target_index))
        standard_spectra[(standard_id, target.standard_rt_isomer, target.nce)] = spectrum

print(
    "Authentic-standard NCE spectra evaluable: "
    f"{sum(value is not None for value in standard_spectra.values())}/{len(standard_spectra)}"
)

authentic_score_rows = []
for pair in ms1_pairs.itertuples(index=False):
    candidate_base_key = (
        pair.steroid,
        pair.derivatization,
        pair.matrix,
        pair.enzyme,
        pair.unlabeled_feature,
    )
    scores = []
    for nce in NCE_VALUES:
        candidate_spectrum = candidate_spectra.get((*candidate_base_key, nce))
        standard_spectrum = standard_spectra.get((
            pair.authentic_standard_id,
            pair.standard_rt_isomer,
            nce,
        ))
        if candidate_spectrum is None or standard_spectrum is None:
            continue
        score = modified_cosine.pair(standard_spectrum, candidate_spectrum)
        scores.append((float(score["score"]), int(score["matches"]), nce))

    if scores:
        best_score, best_matches, best_nce = max(scores, key=lambda item: (item[0], item[1]))
    else:
        best_score, best_matches, best_nce = np.nan, np.nan, np.nan
    authentic_score_rows.append({
        **pair._asdict(),
        "authentic_standard_modified_cosine": best_score,
        "authentic_standard_best_nce": best_nce,
        "authentic_standard_matched_peaks": best_matches,
    })

scored_isomers = pd.DataFrame(authentic_score_rows)
same_standard_key = CANDIDATE_KEY + ["authentic_standard_id"]
scored_isomers["_score_sort"] = scored_isomers["authentic_standard_modified_cosine"].fillna(-np.inf)
scored_isomers["_rt_sort"] = scored_isomers["rt_error_s"].abs()
best_annotations = (
    scored_isomers
    .sort_values(
        ["_score_sort", "authentic_standard_matched_peaks", "_rt_sort"],
        ascending=[False, False, True],
    )
    .drop_duplicates(same_standard_key, keep="first")
    .drop(columns=["_score_sort", "_rt_sort", "standard_rt_isomer"])
)

## 9. Apply authentic-standard MS2 and RT-disambiguation rules

In [ ]:
annotation_columns = [
    "authentic_standard_id", "authentic_standard_name", "KEGG_number",
    "mz_error_ppm", "rt_error_s", "authentic_standard_modified_cosine",
    "authentic_standard_best_nce", "authentic_standard_matched_peaks",
]
all_annotations = parent_output.merge(
    best_annotations[CANDIDATE_KEY + annotation_columns],
    on=CANDIDATE_KEY,
    how="inner",
    validate="one_to_many",
).rename(columns={"best_modified_cosine": "candidate_parent_modified_cosine"})

authentic_ms2_pass = (
    all_annotations["authentic_standard_modified_cosine"].gt(AUTHENTIC_MIN_MODIFIED_COSINE)
    & all_annotations["authentic_standard_matched_peaks"].ge(AUTHENTIC_MIN_MATCHED_PEAKS)
)
ms2_filtered = all_annotations.loc[authentic_ms2_pass].copy()

keep_indices = []
for _, group in ms2_filtered.groupby(CANDIDATE_KEY, sort=False, dropna=False):
    abs_rt_error = group["rt_error_s"].abs()
    unresolved = abs_rt_error.le(AUTHENTIC_UNRESOLVED_RT_S)
    if unresolved.any():
        keep_indices.extend(group.index[unresolved].tolist())
    else:
        minimum_error = abs_rt_error.min()
        keep_indices.extend(group.index[np.isclose(abs_rt_error, minimum_error, atol=1e-9)].tolist())

authentic_columns = [
    "steroid", "derivatization", "matrix", "enzyme",
    "unlabeled_feature", "unlabeled_feature_mz", "unlabeled_feature_rtime",
    "candidate_parent_modified_cosine",
    "authentic_standard_id", "authentic_standard_name", "KEGG_number",
    "mz_error_ppm", "rt_error_s", "authentic_standard_modified_cosine",
    "authentic_standard_best_nce", "authentic_standard_matched_peaks",
]
authentic_output = (
    ms2_filtered.loc[keep_indices, authentic_columns]
    .sort_values(CANDIDATE_KEY + ["authentic_standard_id"])
    .reset_index(drop=True)
)
authentic_path = RESULT_DIR / "02_authentic_standard_annotation.tsv"
authentic_output.to_csv(authentic_path, sep="\t", index=False)

print(f"Wrote: {authentic_path.relative_to(PROJECT_DIR)}")
print(f"MS1-matched annotation rows before MS2 filtering: {len(all_annotations):,}")
print(f"Rows passing authentic MS2 criteria: {len(ms2_filtered):,}")
print(f"Rows after RT disambiguation: {len(authentic_output):,}")
print(f"Unique authentic-standard-identified candidate keys: {len(authentic_output[CANDIDATE_KEY].drop_duplicates()):,}")
display(authentic_output.head(10))

## 10. Integrate both annotation types into the complete v9 candidate table

In [ ]:
parent_annotation_keys = set(
    parent_output.loc[parent_output["ms2_supported_any_nce"], CANDIDATE_KEY]
    .itertuples(index=False, name=None)
)
authentic_annotation_keys = set(
    authentic_output[CANDIDATE_KEY].itertuples(index=False, name=None)
)

pathway_output = base_candidates[[
    "steroid", "derivatization", "matrix", "enzyme",
    "unlabeled_feature", "unlabeled_feature_mz", "unlabeled_feature_rtime",
]].copy()
pathway_keys = list(pathway_output[CANDIDATE_KEY].itertuples(index=False, name=None))
pathway_output["authentic_standard_identification"] = [
    key in authentic_annotation_keys for key in pathway_keys
]
pathway_output["parent_candidate_annotation"] = [
    key in parent_annotation_keys for key in pathway_keys
]
pathway_output = pathway_output.sort_values(CANDIDATE_KEY).reset_index(drop=True)

pathway_path = RESULT_DIR / "03_CYP_steroid_feature_annotations.tsv"
pathway_output.to_csv(pathway_path, sep="\t", index=False)
legacy_pathway_path = RESULT_DIR / "03_pathway_comparison_input.tsv"
legacy_pathway_path.unlink(missing_ok=True)

summary = pd.DataFrame([{
    "total_candidates": len(pathway_output),
    "parent_candidate_annotation": int(pathway_output["parent_candidate_annotation"].sum()),
    "authentic_standard_identification": int(pathway_output["authentic_standard_identification"].sum()),
    "both": int((
        pathway_output["parent_candidate_annotation"]
        & pathway_output["authentic_standard_identification"]
    ).sum()),
    "parent_only": int((
        pathway_output["parent_candidate_annotation"]
        & ~pathway_output["authentic_standard_identification"]
    ).sum()),
    "authentic_only": int((
        ~pathway_output["parent_candidate_annotation"]
        & pathway_output["authentic_standard_identification"]
    ).sum()),
    "neither": int((
        ~pathway_output["parent_candidate_annotation"]
        & ~pathway_output["authentic_standard_identification"]
    ).sum()),
}])

print(f"Wrote: {pathway_path.relative_to(PROJECT_DIR)}")
display(summary)
display(pathway_output.head(10))

## 11. Final reproducibility checks

In [ ]:
assert len(parent_output) == len(base_candidates)
assert len(pathway_output) == len(base_candidates)
assert parent_output["enzyme"].map(lambda value: value == "WT" or value.startswith("CYP")).all()
assert authentic_output["enzyme"].map(lambda value: value == "WT" or value.startswith("CYP")).all()
assert pathway_output["enzyme"].map(lambda value: value == "WT" or value.startswith("CYP")).all()
assert not parent_output.duplicated(CANDIDATE_KEY).any()
assert not pathway_output.duplicated(CANDIDATE_KEY).any()
assert set(pathway_output[CANDIDATE_KEY].itertuples(index=False, name=None)) == set(
    parent_output[CANDIDATE_KEY].itertuples(index=False, name=None)
)
assert set(authentic_annotation_keys).issubset(
    set(pathway_output[CANDIDATE_KEY].itertuples(index=False, name=None))
)

expected_outputs = [
    RESULT_DIR / "01_parent_candidate_comparison.tsv",
    RESULT_DIR / "02_authentic_standard_annotation.tsv",
    RESULT_DIR / "03_CYP_steroid_feature_annotations.tsv",
]
assert all(path.exists() for path in expected_outputs)
assert not (RESULT_DIR / "04_annotation_summary.tsv").exists()

print("All reproducibility checks passed.")
for path in expected_outputs:
    print(f"- {path.relative_to(PROJECT_DIR)}")